# Hierarchy Engine Engineer Demo

This notebook is a practical walkthrough of the `hierarchy_engine` project.

The goal is not just to show happy-path usage. By the end, an engineer should understand:

- the object model
- the YAML authoring format
- each validation stage and why it exists
- how flattening works
- how version comparison works
- how publish protection works against Spark tables
- when post-publish validation is useful


## Learning Objectives

This notebook is organized as a sequence of increasingly realistic steps:

1. Load several YAML hierarchies.
2. Inspect tolerant load issues.
3. Run pre-structural validation against the nested object model.
4. Render and flatten a hierarchy.
5. Run post-structural validation against the flattened rows.
6. Compare two hierarchy versions.
7. Export a hierarchy back to YAML.
8. Run a full publish flow against demo Spark tables.
9. Re-run pre-publish validation to see how duplicate-version protection works.
10. Run post-publish validation as an audit step.

The supporting YAML files for this notebook live under `examples/`:

- `demo_baseline_hierarchy.yaml`
- `demo_modified_hierarchy.yaml`
- `demo_invalid_hierarchy.yaml`


In [ ]:
from dataclasses import asdict, replace
from pathlib import Path
import re

import pandas as pd

from hierarchy_engine.post_structural_validator import PostStructuralHierarchyValidator
from hierarchy_engine.service import HierarchyService

service = HierarchyService()
project_root = Path.cwd()
examples_dir = project_root / "examples"

baseline_path = examples_dir / "demo_baseline_hierarchy.yaml"
modified_path = examples_dir / "demo_modified_hierarchy.yaml"
invalid_path = examples_dir / "demo_invalid_hierarchy.yaml"

for path in (baseline_path, modified_path, invalid_path):
    print(path, "exists=", path.exists())

In [ ]:
def print_validation_result(title, result):
    print(f"\n=== {title} ===")
    print(result.to_text())


def issues_to_frame(issues):
    return pd.DataFrame(
        [
            {
                "severity": issue.severity,
                "check_name": issue.check_name,
                "message": issue.message,
                "details": issue.details,
            }
            for issue in issues
        ]
    )


def rows_to_frame(rows):
    return pd.DataFrame([asdict(row) for row in rows])


## 1. Load Demo Hierarchies

The loader is intentionally tolerant. It tries to build a usable `HierarchyDefinition` while collecting field-level parse issues in `definition.load_issues`.

That design matters because engineers and future UI users need to see *what is wrong* rather than just getting a hard parse failure for every malformed field.

In [ ]:
baseline_definition = service.load_from_yaml(baseline_path)
modified_definition = service.load_from_yaml(modified_path)
invalid_definition = service.load_from_yaml(invalid_path)

print("Baseline metadata:")
print(baseline_definition.metadata)

print("\nModified metadata:")
print(modified_definition.metadata)

print("\nInvalid hierarchy load issues:")
issues_to_frame(invalid_definition.load_issues)

## 2. Pre-Structural Validation

This stage validates the nested in-memory hierarchy *before* flattening.

Typical checks include:

- missing metadata
- invalid version status
- invalid date ranges
- duplicate `account_key`
- malformed node objects
- cycles
- empty root collection


In [ ]:
baseline_validation = service.get_validation_result(baseline_definition)
invalid_validation = service.get_validation_result(invalid_definition)

print_validation_result("Baseline Pre-Structural Validation", baseline_validation)
print_validation_result("Invalid Pre-Structural Validation", invalid_validation)

issues_to_frame(invalid_validation.issues)

## 3. Render and Flatten

The hierarchy is authored as a nested tree, but persisted as adjacency-list rows.

This section makes that transformation concrete.

In [ ]:
print(service.render_tree(baseline_definition))

In [ ]:
baseline_rows = service.flatten_definition(baseline_definition)
rows_to_frame(baseline_rows).sort_values(["account_level", "node_path"]).reset_index(drop=True)

## 4. Post-Structural Validation

This stage validates the flattened rows before any Spark write.

Why this stage exists:

- some problems are easier to detect in the flattened artifact than in the nested tree
- this is where path/level/parent consistency really becomes visible
- it protects publish from malformed adjacency-list rows


In [ ]:
baseline_post_structural = service.get_post_structural_validation_result(baseline_definition)
print_validation_result("Baseline Post-Structural Validation", baseline_post_structural)

In [ ]:
corrupted_rows = list(baseline_rows)
corrupted_rows[3] = replace(
    corrupted_rows[3],
    parent_account_key="99999",
    node_path="BROKEN||PATH",
    account_level=99,
)

manual_post_structural = PostStructuralHierarchyValidator().validate_rows(
    metadata=baseline_definition.metadata,
    rows=corrupted_rows,
)

print_validation_result("Manually Corrupted Post-Structural Validation", manual_post_structural)
issues_to_frame(manual_post_structural.issues)

## 5. Compare Two Hierarchy Versions

The comparer operates on the flattened representation and currently detects:

- added nodes
- removed nodes
- renamed nodes
- reparented nodes

The modified demo hierarchy intentionally includes examples of all of these.

In [ ]:
diff = service.compare_definitions(baseline_definition, modified_definition)
print(service.render_diff(baseline_definition, modified_definition))

In [ ]:
pd.DataFrame([
    {
        "change_type": item.change_type,
        "account_key": item.account_key,
        "old_value": item.old_value,
        "new_value": item.new_value,
    }
    for item in diff.items
])

## 6. Export Back to YAML

The exporter demonstrates that the in-memory object model can be serialized back to YAML without having to rely on the original file text.

In [ ]:
exported_yaml = service.export_to_yaml(baseline_definition)
print("\n".join(exported_yaml.splitlines()[:25]))

## 7. Live Spark Demo: Pre-Publish Validation, Publish, Duplicate Protection, Post-Publish Audit

This section uses real Spark tables.

What it demonstrates:

- clean pre-publish validation against empty or absent tables
- publish through the service
- duplicate-version blocking on a second publish attempt
- post-publish audit validation against persisted tables

The notebook will create and drop demo tables in the current schema using a user-specific prefix.

In [ ]:
current_db = spark.catalog.currentDatabase()
current_user = spark.sql("SELECT current_user() AS user_name").first()["user_name"]
safe_user = re.sub(r"[^A-Za-z0-9_]", "_", current_user).lower()
prefix = f"{safe_user}_hierarchy_engine_demo"

registry_table = f"{current_db}.{prefix}_registry"
version_table = f"{current_db}.{prefix}_version"
node_table = f"{current_db}.{prefix}_nodes"

print("Current database:", current_db)
print("Registry table:", registry_table)
print("Version table:", version_table)
print("Node table:", node_table)

for table_name in (node_table, version_table, registry_table):
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")

print("Demo tables dropped if they previously existed.")

In [ ]:
clean_pre_publish = service.get_pre_publish_validation_result(
    definition=baseline_definition,
    spark=spark,
    registry_table=registry_table,
    version_table=version_table,
    node_table=node_table,
)

print_validation_result("Pre-Publish Validation Against Clean State", clean_pre_publish)

In [ ]:
publish_definition = service.load_from_yaml(baseline_path)
publish_definition.metadata.version_status = "published"

service.publish_to_tables(
    definition=publish_definition,
    spark=spark,
    registry_table=registry_table,
    version_table=version_table,
    node_table=node_table,
    node_write_mode="append",
    created_by=current_user,
    published_by=current_user,
    change_description="Engineer demo publish",
)

print("Publish completed.")

In [ ]:
print("Registry rows")
spark.table(registry_table).show(truncate=False)

print("Version rows")
spark.table(version_table).show(truncate=False)

print("Node rows")
spark.table(node_table).orderBy("node_path").show(truncate=False)

In [ ]:
duplicate_pre_publish = service.get_pre_publish_validation_result(
    definition=publish_definition,
    spark=spark,
    registry_table=registry_table,
    version_table=version_table,
    node_table=node_table,
)

print_validation_result("Pre-Publish Validation After Publish", duplicate_pre_publish)
issues_to_frame(duplicate_pre_publish.issues)

In [ ]:
post_publish_audit = service.validate_published_version(
    spark=spark,
    hierarchy_id=publish_definition.metadata.hierarchy_id,
    version_id=publish_definition.metadata.version_id,
    node_table=node_table,
    version_table=version_table,
)

print_validation_result("Post-Publish Audit Validation", post_publish_audit)

## 8. Cleanup

Run the next cell if you want to remove the demo tables after the walkthrough.

In [ ]:
for table_name in (node_table, version_table, registry_table):
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")

print("Demo tables removed.")

## Suggested Engineer Exercises

To build real fluency, try the following after completing the notebook:

1. Add a new node to the baseline hierarchy and inspect the comparer output.
2. Introduce a duplicate `account_key` in the YAML and observe pre-structural validation.
3. Manually corrupt a flattened row's `node_path` and observe post-structural validation.
4. Publish a `published` version, then try publishing another overlapping `published` version and inspect pre-publish validation.
5. Change registry metadata such as `owner_team` for an existing hierarchy and inspect the registry conflict checks.
6. Create a `retired` version and reason through how your operational workflow should transition current versions.

If an engineer can explain the outcome of those six exercises without guessing, they understand the project at the right depth.